# 20260716_EDA_2024_전국_시도분리정제

- 작성자: 이정연
- #5 이슈 참고 
- 공통 함수는 일단 따로 안만들고, 이 노트북에서 진행해보고 결정해서 유틸로 분리하는 방식으로 진행

In [1]:
# 라이브러리 임포트
import re
from pathlib import Path

import pandas as pd
import numpy as np

# 경로 설정
DATA_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")
REPORTS_DIR = Path("../reports")

# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시

In [2]:
# 데이터 로드
file_2024 = (
    DATA_DIR
    / "칼럼정렬"
    / "세부사업표추출_2024년도 지방자치단체 저출산고령사회 시행계획 (제4차 기본계획)_칼럼정렬.xlsx"
)
df_raw = pd.read_excel(file_2024, sheet_name="정리본_자동")

print(df_raw.shape)
df_raw.head(10)

(7340, 12)


,지역,세부사업명,사업분류재정구분,2024년 예산,2023년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,Ⅰ. 공통사업,NaN,3906937,3952421.7,45485,1,NaN,1,3,5,데이터행
1,서울,1. 함께 일하고 함께 돌보는 사회(공통),공통,3260099,3058321,201778,7,NaN,1,3,6,데이터행
2,서울,저출생 극복 지역네트워크 구축사업 지원,공통,54,108,-54,-50,지원대상 : 서울시민(3~7세 자녀가 있는 가정)\n지원내용:서울100인의아빠단,1,3,7,데이터행
3,서울,입양아동 가족지원,공통,4002,4345,-343,-8,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용:입양아동양육보조금,양육수당,입양비용등",1,3,8,데이터행
4,서울,국공립어린이집 등 보육서비스\n수준제고,공통,459506,450562,8944,2,지원대상 : 국공립 어린이집 등 지원내용:보육교직원인건비지원,1,3,9,데이터행
5,서울,어린이집 이용 영유아 무상보육 확대,공통,554805,654969,-100164,-15,지원대상 : 만0~2세 영아 지원내용:어린이집이용영아대상바우처지원,1,3,10,데이터행
6,서울,초등돌봄 공적\n인프라 확충,공통,1141,5087,-3946,-78,지원대상 : 6~12세 아동\n지원내용:돌봄서비스제공,1,3,11,데이터행
7,서울,육아종합지원센터 운영,공통,2244,2476,-232,-9,"지원대상 : 어린이집 및 영유아 양육 가정 지원내용\n어린이집 지원사업 : 어린이집 맞춤 컨설팅, 장애아지원 교육 보육교직원 교육 및 상담, 서울형 모아어린이집 운영지원 등\n가정양육 지원사업 : 장난감도서관 운영, 시\n간제보육 관리등",1,3,12,데이터행
8,서울,가정양육수당 지원,공통,44151,55556,-11405,-21,지원대상 : 시설 미이용 24개월~86개월 미만 아동 지원내용:월10만원(장애아동35개월미만20만원) 지원,1,3,13,데이터행
9,서울,부모급여,공통,688871,388889,299982,77,"지원대상 : 0~1세 아동\n지원내용:0세100만원(월),1세50만원(월)지원",1,3,14,데이터행


In [3]:
# 기본검사
print("[데이터셋 크기]\n", df_raw.shape)
print("=====================================")
print("[데이터 타입]\n", df_raw.dtypes)
print("=====================================")
print("[각 컬럼별 결측치 개수 평균]\n", df_raw.isna().mean().round(3))
print("=====================================")
print("[중복 행 수]", df_raw.duplicated().sum())
print("=====================================")

# 지역(시도) 목록 확인
print("[지역(시도) 목록 확인]\n", df_raw["지역"].value_counts())

[데이터셋 크기]
 (7340, 12)
[데이터 타입]
 지역             str
세부사업명          str
사업분류재정구분       str
2024년 예산    object
2023년 예산    object
증감액         object
비율          object
주요내용           str
원본표구간        int64
머리글행         int64
원본행          int64
행유형            str
dtype: object
[각 컬럼별 결측치 개수 평균]
 지역          0.000
세부사업명       0.000
사업분류재정구분    0.023
2024년 예산    0.002
2023년 예산    0.004
증감액         0.002
비율          0.005
주요내용        0.041
원본표구간       0.000
머리글행        0.000
원본행         0.000
행유형         0.000
dtype: float64
[중복 행 수] 0
[지역(시도) 목록 확인]
 지역
경기    1326
경남     725
경북     705
부산     694
충북     512
강원     507
전남     479
전북     430
인천     339
울산     313
대구     264
서울     247
광주     211
대전     173
제주     167
세종     131
충남     117
Name: count, dtype: int64


In [4]:
# 지역별로 데이터 분리
sido_dfs = {sido: group.copy() for sido, group in df_raw.groupby("지역")}

# 시도별 행 수 확인
for sido, df_sido in sido_dfs.items():
    print(sido, len(df_sido))

강원 507
경기 1326
경남 725
경북 705
광주 211
대구 264
대전 173
부산 694
서울 247
세종 131
울산 313
인천 339
전남 479
전북 430
제주 167
충남 117
충북 512


In [5]:
# 서울만 따로 확인 - #3 산출물 검증
df_seoul = sido_dfs["서울"]

print(df_seoul.shape)
df_seoul.head(10)

(247, 12)


,지역,세부사업명,사업분류재정구분,2024년 예산,2023년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,Ⅰ. 공통사업,NaN,3906937,3952421.7,45485,1,NaN,1,3,5,데이터행
1,서울,1. 함께 일하고 함께 돌보는 사회(공통),공통,3260099,3058321,201778,7,NaN,1,3,6,데이터행
2,서울,저출생 극복 지역네트워크 구축사업 지원,공통,54,108,-54,-50,지원대상 : 서울시민(3~7세 자녀가 있는 가정)\n지원내용:서울100인의아빠단,1,3,7,데이터행
3,서울,입양아동 가족지원,공통,4002,4345,-343,-8,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용:입양아동양육보조금,양육수당,입양비용등",1,3,8,데이터행
4,서울,국공립어린이집 등 보육서비스\n수준제고,공통,459506,450562,8944,2,지원대상 : 국공립 어린이집 등 지원내용:보육교직원인건비지원,1,3,9,데이터행
5,서울,어린이집 이용 영유아 무상보육 확대,공통,554805,654969,-100164,-15,지원대상 : 만0~2세 영아 지원내용:어린이집이용영아대상바우처지원,1,3,10,데이터행
6,서울,초등돌봄 공적\n인프라 확충,공통,1141,5087,-3946,-78,지원대상 : 6~12세 아동\n지원내용:돌봄서비스제공,1,3,11,데이터행
7,서울,육아종합지원센터 운영,공통,2244,2476,-232,-9,"지원대상 : 어린이집 및 영유아 양육 가정 지원내용\n어린이집 지원사업 : 어린이집 맞춤 컨설팅, 장애아지원 교육 보육교직원 교육 및 상담, 서울형 모아어린이집 운영지원 등\n가정양육 지원사업 : 장난감도서관 운영, 시\n간제보육 관리등",1,3,12,데이터행
8,서울,가정양육수당 지원,공통,44151,55556,-11405,-21,지원대상 : 시설 미이용 24개월~86개월 미만 아동 지원내용:월10만원(장애아동35개월미만20만원) 지원,1,3,13,데이터행
9,서울,부모급여,공통,688871,388889,299982,77,"지원대상 : 0~1세 아동\n지원내용:0세100만원(월),1세50만원(월)지원",1,3,14,데이터행


In [6]:
# 이전 데이터셋에서 확인한 classify_row 적용
sido_title_pattern = re.compile(r"붙\s*임\s*\(([^)]+)\)")


def classify_row(세부사업명: str) -> str:
    """대분류_소계 / 중분류_소계 / 헤더반복 / 세부사업 행 판별"""
    if pd.isna(세부사업명) or str(세부사업명).strip() == "":
        return "헤더반복"

    name = str(세부사업명).strip()

    if name == "세부사업명":
        return "헤더반복"
    if sido_title_pattern.search(name):
        return "헤더반복"
    if re.match(
        r"^[Ⅰ-Ⅿ]", name
    ):  # 유니코드 로마숫자 대문자 전체 블록(Ⅰ~ⅬⅭⅮⅯ) - 대분류 10개(Ⅹ) 초과 대비
        return "대분류_소계"
    if re.match(r"^\d+\.", name) and re.search(r"\((공통|자체)\)$", name):  # 조건 추가
        return "중분류_소계"
    return "세부사업"


df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)
df_raw["2024년 예산"] = pd.to_numeric(
    df_raw["2024년 예산"].astype(str).str.replace(",", "").replace("-", 0), errors="coerce"
)

-> #3의 결과와 정확하게 일치하는 것 확인

-> 이 노트북에서 서울은 따로 데이터셋 정제 불필요

In [7]:
df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)

# 시도별 행종류 분포
rowtype_distribution = df_raw.groupby(["지역", "사업행구분"]).size().unstack(fill_value=0)
rowtype_distribution

사업행구분,대분류_소계,세부사업,중분류_소계,헤더반복
지역,,,,
강원,2,496,8,1
경기,2,1315,8,1
경남,2,714,8,1
경북,2,694,8,1
광주,2,200,8,1
대구,2,253,8,1
대전,2,162,8,1
부산,2,683,8,1
서울,2,237,8,0


-> 서울은 0건, 나머지 16개 시도는 시도경계 제목행("붙임 (OO시 OO교육청)") 1건씩 확인

-> 정리본_자동 sheet에서 정제된 것 확인 

-> 대분류_소계도 2개로 일관적임을 확인

-> 경기 경남에서 중분류_소계가 9건이라 확인 필요

In [8]:
# 경기 중분류_소계 행 확인
df_gyeonggi = sido_dfs["경기"]
df_gyeonggi["사업행구분"] = df_gyeonggi["세부사업명"].apply(classify_row)
print("[경기] 중분류_소계 행 확인")
print("==================================================")
print(df_gyeonggi[df_gyeonggi["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

print("\n")

# 경남도 동일하게 확인
df_gyeongnam = sido_dfs["경남"]
df_gyeongnam["사업행구분"] = df_gyeongnam["세부사업명"].apply(classify_row)
print("[경남] 중분류_소계 행 확인")
print("==================================================")
print(df_gyeongnam[df_gyeongnam["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

[경기] 중분류_소계 행 확인
2374      1. 함께 일하고 함께 돌보는 사회(공통)
2414       2. 건강하고 능동적인\n고령사회(공통)
2423    3. 모두의 역량이 고루 발휘되는 사회(공통)
2436       4.인구구조\n변화에 대한\n적응(공통)
2443      1. 함께 일하고 함께 돌보는 사회(자체)
3030     2. 건강하고 능동적인 고령사회 구축(자체)
3214    3. 모두의 역량이 고루 발휘되는 사회(자체)
3519         4.인구구조 변화에 대한 적응(자체)
Name: 세부사업명, dtype: str


[경남] 중분류_소계 행 확인
6450      1. 함께 일하고 함께 돌보는 사회(공통)
6503       2. 건강하고 능동적인\n고령사회(공통)
6515    3. 모두의 역량이 고루 발휘되는 사회(공통)
6526         4.인구구조 변화에 대한 적응(공통)
6531      1. 함께 일하고 함께 돌보는 사회(자체)
6813    2. 건강하고 능동적인 고령사회\n구축(자체)
6929    3. 모두의 역량이 고루 발휘되는 사회(자체)
7079         4.인구구조 변화에 대한 적응(자체)
Name: 세부사업명, dtype: str


-> 진짜 중분류_소계 행들을 보면 뒤에 (공통) / (자체) 가 오는 것을 확인할 수 있음. 

-> 이  끝부분 조건 추가하면 오분류 걸러내기 가능함. 

-> classify_row 함수 수정 

-> 조건 추가하니까 해결됨

In [9]:
# 지역별로 원본 순서 유지한 채 대분류/중분류 라벨 세부사업행에 전파
def assign_labels(df_sido: pd.DataFrame) -> pd.DataFrame:
    """대분류_소계 / 중분류_소계 행의 이름을 뒤따르는 세부사업 행에 전파"""
    df_sido = df_sido.sort_values(
        "원본행"
    ).copy()  # 그냥 copy해도 되지만(raw도 이미 순서가 정렬되어있음) ffill이 순서에 의존하므로 원본 문서 순서를 명시적으로 보장함.
    major_mask = df_sido["사업행구분"] == "대분류_소계"
    medium_mask = df_sido["사업행구분"] == "중분류_소계"
    df_sido["대분류"] = df_sido["세부사업명"].where(major_mask).ffill()
    df_sido["중분류"] = df_sido["세부사업명"].where(medium_mask).ffill()

    return df_sido


df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]

# leaf(세부사업)만 추출
df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_leaf.shape)
print(df_leaf.columns.tolist())
df_leaf.head()

(7154, 15)
['세부사업명', '사업분류재정구분', '2024년 예산', '2023년 예산', '증감액', '비율', '주요내용', '원본표구간', '머리글행', '원본행', '행유형', '사업행구분', '대분류', '중분류', '지역']


,세부사업명,사업분류재정구분,2024년 예산,2023년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,대분류,중분류,지역
2,저출생 극복 지역네트워크 구축사업 지원,공통,54.0,108,-54,-50,지원대상 : 서울시민(3~7세 자녀가 있는 가정)\n지원내용:서울100인의아빠단,1,3,7,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
3,입양아동 가족지원,공통,4002.0,4345,-343,-8,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용:입양아동양육보조금,양육수당,입양비용등",1,3,8,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
4,국공립어린이집 등 보육서비스\n수준제고,공통,459506.0,450562,8944,2,지원대상 : 국공립 어린이집 등 지원내용:보육교직원인건비지원,1,3,9,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
5,어린이집 이용 영유아 무상보육 확대,공통,554805.0,654969,-100164,-15,지원대상 : 만0~2세 영아 지원내용:어린이집이용영아대상바우처지원,1,3,10,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
6,초등돌봄 공적\n인프라 확충,공통,1141.0,5087,-3946,-78,지원대상 : 6~12세 아동\n지원내용:돌봄서비스제공,1,3,11,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울


In [10]:
mask_non_numeric = (
    pd.to_numeric(df_leaf["2024년 예산"], errors="coerce").isna() & df_leaf["2024년 예산"].notna()
)
display(df_leaf.loc[mask_non_numeric, ["지역", "세부사업명", "2024년 예산"]])

,지역,세부사업명,2024년 예산


In [11]:
# QA: 중분류_소계 행 예산 vs leaf 합계 대조 (17개 시도 x 중분류 단위)
leaf_sum = (
    df_leaf.groupby(["지역", "중분류"])["2024년 예산"]
    .sum()
    .reset_index()
    .rename(columns={"2024년 예산": "leaf_합계"})
)

# 원본 소계 행 값 (중분류_소계 행에서 직접 가져온 값)
subtotal = df_labeled[df_labeled["사업행구분"] == "중분류_소계"][
    ["지역", "대분류", "세부사업명", "2024년 예산"]
].rename(columns={"세부사업명": "중분류", "2024년 예산": "원본_소계값"})

qa = subtotal.merge(leaf_sum, on=["지역", "중분류"], how="left")
qa["결과"] = np.where(qa["leaf_합계"] == qa["원본_소계값"], "일치", "불일치")
qa["결과"].value_counts()

결과
일치     114
불일치     22
Name: count, dtype: int64

In [12]:
qa.head()

,지역,대분류,중분류,원본_소계값,leaf_합계,결과
0,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),3260099.0,3260099.0,일치
1,서울,Ⅰ. 공통사업,2. 건강하고 능동적인\n고령사회(공통),84128.0,84128.0,일치
2,서울,Ⅰ. 공통사업,3. 모두의 역량이 고루 발휘되는\n사회(공통),458096.0,458096.0,일치
3,서울,Ⅰ. 공통사업,4.인구구조 변화에 대한 적응(공통),104614.0,104614.0,일치
4,서울,Ⅱ. 지자체\n자체사업,1. 함께 일하고 함께 돌보는 사회(자체),483598.0,483598.0,일치


In [13]:
qa["차이"] = qa["leaf_합계"] - qa["원본_소계값"]

# 결과 컬럼을 항상 채워서, 저장 파일이 비어있어도 "검증은 했다"는 게 남도록 함
qa["결과"] = qa["차이"].abs().le(0.01).map({True: "일치", False: "불일치"})

qa_mismatch = qa[qa["결과"] == "불일치"]
print(f"검증 대상: {len(qa)}건 / 불일치: {len(qa_mismatch)}건")

검증 대상: 136건 / 불일치: 22건


In [14]:
display(qa_mismatch[["지역", "대분류", "중분류", "원본_소계값", "leaf_합계", "차이"]])

,지역,대분류,중분류,원본_소계값,leaf_합계,차이
12,부산,Ⅱ. 지자체\n자체사업,1. 함께 일하고\n함께 돌보는\n사회(자체),511612.0,511613.0,1.0
14,부산,Ⅱ. 지자체\n자체사업,3. 모두의 역량이 고루 발휘되는\n사회(자체),58329.0,58331.0,2.0
35,광주,Ⅰ. 공통사업,4.인구구조 변화에 대한 적응(공통),0.0,NaN,NaN
68,경기,Ⅱ. 지자체\n자체사업,1. 함께 일하고 함께 돌보는 사회(자체),1019222.0,1019224.0,2.0
70,경기,Ⅱ. 지자체\n자체사업,3. 모두의 역량이 고루 발휘되는 사회(자체),425962.0,425965.0,3.0
84,충북,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),121958.0,121959.0,1.0
85,충북,Ⅱ. 지자체 자체사업,2. 건강하고 능동적인 고령사회\n구축(자체),36615.0,36616.0,1.0
87,충북,Ⅱ. 지자체 자체사업,4.인구구조 변화에 대한 적응(자체),116980.0,116979.0,-1.0
88,충남,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),522929.0,411451.0,-111478.0
89,충남,Ⅰ. 공통사업,2. 건강하고 능동적인\n고령사회(공통),617588.0,21136.0,-596452.0


# 충남 QA 불일치 원인 확인 - 원본 Table 1 대조

`정리본_자동`에서 충남만 QA 불일치가 크게 나온 원인이 `정리본_자동` 변환 과정의 데이터 손실인지, 원본 문서 자체의 문제인지 확인한다.

`원본행` 컬럼은 `Table 1` 시트의 1-indexed 엑셀 행 번호를 그대로 가리키므로, 이 값으로 원본 위치를 바로 대조할 수 있다.

In [15]:
df_chungnam_raw = sido_dfs["충남"]
mask_nan = pd.to_numeric(
    df_chungnam_raw["2024년 예산"].astype(str).str.replace(",", "").replace("-", 0), errors="coerce"
).isna()

display(df_chungnam_raw.loc[mask_nan, ["세부사업명", "2024년 예산"]])

,세부사업명,2024년 예산
4717,붙임 (충청남도 충청남도교육청),붙임 (충청남도 충청남도교육청)


In [16]:
df_chungnam = df_labeled[df_labeled["지역"] == "충남"]
print(df_chungnam["사업행구분"].value_counts())
display(df_chungnam[df_chungnam["사업행구분"] == "중분류_소계"]["세부사업명"])

사업행구분
세부사업      106
중분류_소계      8
대분류_소계      2
헤더반복        1
Name: count, dtype: int64


4719      1. 함께 일하고 함께 돌보는 사회(공통)
4759       2. 건강하고 능동적인\n고령사회(공통)
4772    3. 모두의 역량이 고루 발휘되는 사회(공통)
4773         4.인구구조 변화에 대한 적응(공통)
4779    1. 함께 일하고\n함께 돌보는\n사회(자체)
4799    2. 건강하고 능동적인 고령사회\n구축(자체)
4815    3. 모두의 역량이 고루 발휘되는 사회(자체)
4827         4.인구구조 변화에 대한 적응(자체)
Name: 세부사업명, dtype: str

In [17]:
df_chungnam_check = df_labeled[df_labeled["지역"] == "충남"][
    ["원본행", "세부사업명", "사업행구분", "대분류", "중분류"]
]

display(df_chungnam_check.head(30))

,원본행,세부사업명,사업행구분,대분류,중분류
4717,5294,붙임 (충청남도 충청남도교육청),헤더반복,NaN,NaN
4718,5298,Ⅰ. 공통사업,대분류_소계,Ⅰ. 공통사업,NaN
4719,5299,1. 함께 일하고 함께 돌보는 사회(공통),중분류_소계,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4720,5300,이동식놀이교실\n운영,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4721,5301,시간제 보육사업 지원,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4722,5302,영유아보육료,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4723,5303,가정양육수당,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4724,5304,부모급여,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4725,5305,여성장애인 출산비용 지원,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4726,5306,분만취약지 지원,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)


In [19]:
# 원본 Table 1 로드
file_2024_original = (
    DATA_DIR / "세부사업표추출_2024년도 지방자치단체 저출산고령사회 시행계획 (제4차 기본계획).xlsx"
)
df_table1 = pd.read_excel(file_2024_original, sheet_name="Table 1", header=None)


def show_table1_around(center_excel_row: int, window: int, label: str):
    """Table 1 원본 시트에서 특정 엑셀 행(1-indexed) 주변을 보여줌"""
    start, end = center_excel_row - window, center_excel_row + window
    print(f"--- {label} (Table1 엑셀행 {start}~{end}) ---")
    # 1,3열은 병합셀로 생긴 빈 열이라 제외, 나머지는 알아보기 쉬운 이름으로 표시
    view = df_table1.iloc[start - 1 : end, [0, 2, 4]].copy()
    view.columns = ["세부사업명", "공통/자체", "예산"]
    display(view)


# 충남 "1. 함께 일하고..." 중분류 안에서 원본행이 연속되지 않는(결번) 구간을 직접 탐지
df_chungnam_cat1 = df_chungnam_check[
    df_chungnam_check["중분류"] == "1. 함께 일하고 함께 돌보는 사회(공통)"
]
원본행_목록 = df_chungnam_cat1["원본행"].tolist()
결번_구간 = [(a, b) for a, b in zip(원본행_목록, 원본행_목록[1:]) if b - a > 1]
print("결번 구간:", 결번_구간)

for 시작, 끝 in 결번_구간:
    show_table1_around((시작 + 끝) // 2, window=(끝 - 시작), label=f"{시작}~{끝} 결번 구간")

# "3. 모두의 역량이 고루 발휘되는 사회(공통)" 소계 행의 실제 원본행을 가져와 그 주변 확인
소계_행 = df_labeled[
    (df_labeled["지역"] == "충남")
    & (df_labeled["세부사업명"].str.contains("모두의 역량", na=False))
    & (df_labeled["사업행구분"] == "중분류_소계")
]
show_table1_around(int(소계_행["원본행"].iloc[0]), window=3, label="3.모두의역량 소계 전후")

결번 구간: [(5312, 5315), (5333, 5336)]
--- 5312~5315 결번 구간 (Table1 엑셀행 5310~5316) ---


,세부사업명,공통/자체,예산
5309,청소년산모 임신출산 의료비\n지원,공통,26
5310,고위험임산부\n의료비 지원,공통,788
5311,기저귀 및 조제분유 지원,공통,4194
5312,세부사업명,사업 분류,2024년 예산
5313,NaN,공통/ 자체,총예산(a)
5314,첫만남이용권 지원,공통,22874
5315,생애초기건강관리 시범사업,공통,6896


--- 5333~5336 결번 구간 (Table1 엑셀행 5331~5337) ---


,세부사업명,공통/자체,예산
5330,저소득층 기저귀\n조제분유 지원,공통,265
5331,산모신생아\n건강관리 지원사업,공통,1446
5332,산후조리도우미 본인부담금 지원사업,공통,335
5333,세부사업명,사업 분류,2024년 예산
5334,NaN,공통/ 자체,총예산(a)
5335,다자녀 맘 산후\n건강관리 지원,공통,76
5336,난임부부 시술비\n지원,공통,630


--- 3.모두의역량 소계 전후 (Table1 엑셀행 5355~5361) ---


,세부사업명,공통/자체,예산
5354,희망소리 찾기 사업(보청기 지원사업),공통,8
5355,세부사업명,사업 분류,2024년 예산
5356,NaN,공통/ 자체,총예산(a)
5357,3. 모두의 역량이 고루 발휘되는 사회(공통),NaN,19928
5358,4.인구구조 변화에 대한 적응(공통),NaN,17795
5359,한부모가족자녀의\n안정적 성장 지원,공통,764
5360,한부모가족\n생활안정 지원,공통,495


# 증감액 부호 소실 문제 확인

팀원 지적: 충남 "3. 모두의 역량이 고루 발휘되는 사회(공통)"는 2024년 예산(19,928)이 2023년 예산(42,867.5)보다 줄었으니 증감액·증감율이 음수여야 하는데, 원본에 양수로 찍혀있다. 이게 충남만의 문제인지, 다른 시도에도 있는지 확인한다.

방법: 세부사업별로 `2024년 예산 - 2023년 예산`을 직접 계산해서, 그 결과가 음수(예산 감소)인 행들만 골라 `증감액` 컬럼이 실제로 음수로 찍혀있는지 대조한다.

In [20]:
def to_numeric_budget(series: pd.Series) -> pd.Series:
    """콤마 제거, '-'는 0으로 처리 후 숫자로 변환 (df_raw의 2024년 예산 변환과 동일한 규칙)"""
    return pd.to_numeric(series.astype(str).str.replace(",", "").replace("-", 0), errors="coerce")


# 이전 셀 실행 여부와 무관하게 독립적으로 동작하도록 2024년 예산도 여기서 다시 변환
df_raw["2024년 예산_num"] = to_numeric_budget(df_raw["2024년 예산"])
df_raw["2023년 예산_num"] = to_numeric_budget(df_raw["2023년 예산"])
df_raw["증감액_num"] = to_numeric_budget(df_raw["증감액"])
df_raw["계산된_증감액"] = df_raw["2024년 예산_num"] - df_raw["2023년 예산_num"]

# 실제로 예산이 감소한 행(계산된 증감액 < 0)만 골라서, 증감액 컬럼 부호를 대조
감소행 = df_raw[df_raw["계산된_증감액"] < -0.5].copy()
감소행["부호_소실"] = 감소행["증감액_num"] > 0.5

print("예산 감소 행 수:", len(감소행))
print("그중 증감액이 양수로 찍힌(부호 소실) 행 수:", 감소행["부호_소실"].sum())

예산 감소 행 수: 2193
그중 증감액이 양수로 찍힌(부호 소실) 행 수: 982


In [21]:
# 지역별로 부호 소실 비율 확인
지역별_부호소실_비율 = (
    감소행.groupby("지역")["부호_소실"].mean().mul(100).round(1).sort_values(ascending=False)
)
지역별_부호소실_비율

지역
인천    100.0
경남    100.0
충남    100.0
대구    100.0
대전    100.0
부산    100.0
전북    100.0
전남     83.3
제주     64.2
서울     61.6
광주      7.9
충북      1.3
울산      0.0
세종      0.0
경기      0.0
경북      0.0
강원      0.0
Name: 부호_소실, dtype: float64

# 증감액 / 비율 직접 재계산
- 위에서 확인했듯이 원본에서는 시도별로 부호나 값 자체가 틀린 경우가 있어 신뢰할 수 없음. 
- 따라서 직접 재계산

In [22]:
def to_numeric_budget(series: pd.Series) -> pd.Series:
    """콤마 제거, '-'는 0으로 처리 후 숫자로 변환 (df_raw의 2024년 예산 변환과 동일한 규칙)"""
    return pd.to_numeric(series.astype(str).str.replace(",", "").replace("-", 0), errors="coerce")


df_leaf["2023년_예산_num"] = to_numeric_budget(df_leaf["2023년 예산"])
df_leaf["2024년_예산_num"] = to_numeric_budget(df_leaf["2024년 예산"])
df_leaf["증감액_재계산"] = df_leaf["2024년_예산_num"] - df_leaf["2023년_예산_num"]
df_leaf["증감율_재계산"] = (
    df_leaf["증감액_재계산"] / df_leaf["2023년_예산_num"].replace(0, np.nan) * 100
).round(1)

df_leaf[
    ["세부사업명", "2024년_예산_num", "2023년_예산_num", "증감액_재계산", "증감율_재계산"]
].head(10)

,세부사업명,2024년_예산_num,2023년_예산_num,증감액_재계산,증감율_재계산
2,저출생 극복 지역네트워크 구축사업 지원,54.0,108.0,-54.0,-50.0
3,입양아동 가족지원,4002.0,4345.0,-343.0,-7.9
4,국공립어린이집 등 보육서비스\n수준제고,459506.0,450562.0,8944.0,2.0
5,어린이집 이용 영유아 무상보육 확대,554805.0,654969.0,-100164.0,-15.3
6,초등돌봄 공적\n인프라 확충,1141.0,5087.0,-3946.0,-77.6
7,육아종합지원센터 운영,2244.0,2476.0,-232.0,-9.4
8,가정양육수당 지원,44151.0,55556.0,-11405.0,-20.5
9,부모급여,688871.0,388889.0,299982.0,77.1
10,공동육아나눔터,2365.0,2136.0,229.0,10.7
11,아이돌봄서비스 확충 및 내실화,96810.0,76220.0,20590.0,27.0


# 최종 스키마

In [23]:
# 텍스트 정리
PUA_PATTERN = re.compile(r"[\uE000-\uF8FF]")


def clean_text(series: pd.Series) -> pd.Series:
    """줄바꿈, 공백을 단일 공백으로 정리"""

    def _clean(x):
        if pd.isna(x):
            return x
        x = PUA_PATTERN.sub("", str(x))
        return re.sub(r"\s+", " ", x).strip()

    return series.apply(_clean)


df_leaf["세부사업명"] = clean_text(df_leaf["세부사업명"])
df_leaf["주요내용"] = clean_text(df_leaf["주요내용"])
df_leaf["대분류"] = clean_text(df_leaf["대분류"])
df_leaf["중분류"] = clean_text(df_leaf["중분류"])

df_leaf.head()

,세부사업명,사업분류재정구분,2024년 예산,2023년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,대분류,중분류,지역,2023년_예산_num,2024년_예산_num,증감액_재계산,증감율_재계산
2,저출생 극복 지역네트워크 구축사업 지원,공통,54.0,108,-54,-50,지원대상 : 서울시민(3~7세 자녀가 있는 가정) 지원내용:서울100인의아빠단,1,3,7,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,108.0,54.0,-54.0,-50.0
3,입양아동 가족지원,공통,4002.0,4345,-343,-8,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용:입양아동양육보조금,양육수당,입양비용등",1,3,8,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,4345.0,4002.0,-343.0,-7.9
4,국공립어린이집 등 보육서비스 수준제고,공통,459506.0,450562,8944,2,지원대상 : 국공립 어린이집 등 지원내용:보육교직원인건비지원,1,3,9,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,450562.0,459506.0,8944.0,2.0
5,어린이집 이용 영유아 무상보육 확대,공통,554805.0,654969,-100164,-15,지원대상 : 만0~2세 영아 지원내용:어린이집이용영아대상바우처지원,1,3,10,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,654969.0,554805.0,-100164.0,-15.3
6,초등돌봄 공적 인프라 확충,공통,1141.0,5087,-3946,-78,지원대상 : 6~12세 아동 지원내용:돌봄서비스제공,1,3,11,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,5087.0,1141.0,-3946.0,-77.6


In [24]:
# 컬럼명 정리 + 연도 추가 + 최종 스키마만 선택
df_leaf_final = (
    df_leaf.drop(columns=["증감액", "비율"])
    .rename(
        columns={
            "2024년_예산_num": "당해예산",
            "2023년_예산_num": "전년도예산",
            "증감액_재계산": "증감액",
            "증감율_재계산": "증감율",
        }
    )
    .assign(연도=2024)
)

df_leaf_final = df_leaf_final[
    [
        "연도",
        "지역",
        "대분류",
        "중분류",
        "사업분류재정구분",
        "세부사업명",
        "주요내용",
        "당해예산",
        "전년도예산",
        "증감액",
        "증감율",
        "원본행",
    ]
]

print(df_leaf_final.shape)
display(df_leaf_final.head())

(7154, 12)


,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,당해예산,전년도예산,증감액,증감율,원본행
2,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,저출생 극복 지역네트워크 구축사업 지원,지원대상 : 서울시민(3~7세 자녀가 있는 가정) 지원내용:서울100인의아빠단,54.0,108.0,-54.0,-50.0,7
3,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,입양아동 가족지원,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용:입양아동양육보조금,양육수당,입양비용등",4002.0,4345.0,-343.0,-7.9,8
4,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립어린이집 등 보육서비스 수준제고,지원대상 : 국공립 어린이집 등 지원내용:보육교직원인건비지원,459506.0,450562.0,8944.0,2.0,9
5,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,어린이집 이용 영유아 무상보육 확대,지원대상 : 만0~2세 영아 지원내용:어린이집이용영아대상바우처지원,554805.0,654969.0,-100164.0,-15.3,10
6,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,초등돌봄 공적 인프라 확충,지원대상 : 6~12세 아동 지원내용:돌봄서비스제공,1141.0,5087.0,-3946.0,-77.6,11


In [25]:
year = 2024

for sido in df_leaf_final["지역"].unique():
    sido_leaf = df_leaf_final[df_leaf_final["지역"] == sido]
    sido_labeled = df_labeled[df_labeled["지역"] == sido]

    sido_dir = INTERIM_DIR / sido
    sido_dir.mkdir(parents=True, exist_ok=True)

    sido_leaf.to_csv(
        sido_dir / f"{year}_{sido}_세부사업_정제.csv", index=False, encoding="utf-8-sig"
    )
    sido_labeled.to_csv(
        sido_dir / f"{year}_{sido}_필터링전_전체원본.csv", index=False, encoding="utf-8-sig"
    )

# QA 결과는 전체 17개 시도 한 파일로 저장 (불일치 포함, 충남 결함도 그대로 남김)
qa.to_csv(REPORTS_DIR / f"{year}_전국_QA_검증결과.csv", index=False, encoding="utf-8-sig")

print("저장 완료: ", df_leaf_final["지역"].nunique(), "개 시도")

저장 완료:  17 개 시도


# 세부사업명 오탈자 / 중복 탐지
- 오탈자를 자동으로 고치지 않고, 사람이 검토할 후보만 찾는다. 
- 같은 지역 / 중분류 안에서 완전히 같지는 않지만 유사도가 높은 세부사업명 쌍을 `rapidfuzz`로 찾는다.

In [26]:
from itertools import combinations
from rapidfuzz import fuzz

In [27]:
def normalize_for_match(name: str) -> str:
    """clean_text로 정제된 세부사업명에서, 비교 목적으로 공백, 문장부호 마저 제거"""
    return re.sub(r"[\s,-./\-/()]", "", name)


def find_near_duplicates(df: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:
    """같은 지역 중분류 안에서 완전 일치는 아니지만 유사도가 높은 세부사업명과 쌍을 찾는다."""
    candidates = []
    for (sido, midium_cat), group in df.groupby(["지역", "중분류"]):
        rows = list(
            group[["세부사업명", "당해예산", "주요내용"]].itertuples(index=False, name=None)
        )
        for (name_a, budget_a, content_a), (name_b, budget_b, content_b) in combinations(rows, 2):
            if name_a == name_b:
                continue
            if normalize_for_match(name_a) == normalize_for_match(name_b):
                continue  # 공백/문장부호 차이 -> 검수 대상 x
            score = fuzz.ratio(name_a, name_b)
            if score >= threshold:
                candidates.append(
                    {
                        "지역": sido,
                        "중분류": midium_cat,
                        "세부사업명1": name_a,
                        "당해예산1": budget_a,
                        "주요내용1": content_a,
                        "세부사업명2": name_b,
                        "당해예산2": budget_b,
                        "주요내용2": content_b,
                        "유사도": score,
                    }
                )
    return pd.DataFrame(candidates).sort_values("유사도", ascending=False)


near_dup = find_near_duplicates(df_leaf_final)
print(len(near_dup), "건 발견")
display(near_dup.head(10))

224 건 발견


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
54,경기,1. 함께 일하고 함께 돌보는 사회(자체),"다자녀 가정, 한부모가정 수도요금 감면",0.0,"다자녀 가구, 한부모 가정 대상으로, 상수도 사 용량 월 최대 10㎥에 해당하는 사용료 감면","다자녀 가정, 한부모가정 하수도요금 감면",0.0,다자녀·한부모 가정 월 사용량 10㎥해당 하수 도사용료 감면,97.674419
139,부산,1. 함께 일하고 함께 돌보는 사회(자체),일생활 균형을 위한 사회적 분위기 확산,2.0,"일생활 균형 교육 및 캠페인 실시, 가족사랑의 날 운영",일·생활 균형을 위한 사회적 분위기 확산,1.0,일·생활 균형(워라밸)을 위한 캠페인 실시 등 홍보 활동,97.674419
3,강원,2. 건강하고 능동적인 고령사회 구축(자체),저소득 노인세대 건강보험료 지원,142.0,"저소득 노인가구 건강, 장기요양보험료 지급",저소득층 노인세대 건강보험료 지원,179.0,영월군에 주소를 둔 국민건강보험공단 지역가입 자로서 건강보험료 부과금액이 '최저보험료 이하 '인 65세 이상 노인세대,97.142857
109,경남,2. 건강하고 능동적인 고령사회 구축(자체),4대 이상 가정 효도수당 지원,10.0,사천시에 3년 이상 주소지를 둔 4대 이상 가정 에 설/명절 효도수당 각 70만원씩 지급,4세대 이상 가정 효도수당 지원,2.0,지원대상 : 거창군에 주소를 두고 4대 이상 함 께 거주하는 가정 지원내용:가구당월10만원지원,96.969697
191,전남,2. 건강하고 능동적인 고령사회 구축(자체),노인 목욕비 및 이미용비 지원,2070.0,노인 목욕비 및 이미용비 지원,노인 목욕 및 이미용비 지원,1887.0,노인목용 및 이미용비 지원,96.774194
159,부산,2. 건강하고 능동적인 고령사회 구축(자체),저소득 노인 건강보험료 지원,60.0,저소득 노인세대 국민건강보험료 지원,저소득층 노인 건강보험료 지원,71.0,보건복지부 장관이 정하여 고시하는 최저보험료 이하 납부하는 65세 이상 노인세대의 건강보험 료 지원,96.774194
131,광주,2. 건강하고 능동적인 고령사회 구축(자체),노인장기요양보험 등급자 지원,70937.0,"저소득층 노인에게 시설급여, 재가급여 지원",노인장기요양보험 등급외자 지원,141.0,실비 입소 등급외자에 대한 시설이용 비용 지 원,96.774194
207,충북,1. 함께 일하고 함께 돌보는 사회(자체),산모·신생아 건강관리 지원,40.0,"지원대상 : 임산부 지원내용:본인부담금의90%지원(상한금50만원) 지원기준:산모가출산(예정)일기준으로6개월전부 터보은군에주민등록을두었으며,신생아출생신고를 군에한경우",산모신생아 건강관리 지원,5249.0,"관내 출산가정에 건강관리사 파견, 서비스 제공",96.296296
219,충북,1. 함께 일하고 함께 돌보는 사회(자체),산모신생아 건강관리사 지원,20.0,"관내 출산가정에 건강관리사 파견, 서비스 제공",산모신생아 건강관리 지원,63.0,지원대상 : 관내 출산부 지원내용 : 본인부담금 발생비용의 90% 지원,96.296296
217,충북,1. 함께 일하고 함께 돌보는 사회(자체),산모신생아 건강관리 지원,5249.0,"관내 출산가정에 건강관리사 파견, 서비스 제공",산모신생아 건강관리사 지원,20.0,"관내 출산가정에 건강관리사 파견, 서비스 제공",96.296296


In [28]:
# 당해예산이 완전히 같은 쌍은 진짜 같은 사업일 가능성이 높은 후보
near_dup_same_budget = near_dup[near_dup["당해예산1"] == near_dup["당해예산2"]]

print(len(near_dup_same_budget), "건 (전체", len(near_dup), "건 중)")
display(near_dup_same_budget)

11 건 (전체 224 건 중)


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
54,경기,1. 함께 일하고 함께 돌보는 사회(자체),"다자녀 가정, 한부모가정 수도요금 감면",0.0,"다자녀 가구, 한부모 가정 대상으로, 상수도 사 용량 월 최대 10㎥에 해당하는 사용료 감면","다자녀 가정, 한부모가정 하수도요금 감면",0.0,다자녀·한부모 가정 월 사용량 10㎥해당 하수 도사용료 감면,97.674419
67,경기,1. 함께 일하고 함께 돌보는 사회(자체),출산가정 수도요금 감면,0.0,NaN,출산가정 하수도요금 감면,0.0,출생신고 한세대 36개월간 하수도요금 감면,96.000000
68,경기,1. 함께 일하고 함께 돌보는 사회(자체),다자녀(세자녀이상) 가구 상수도 요금 감면,0.0,NaN,다자녀(세자녀이상) 가구 하수도 요금 감면,0.0,미성년 3자녀를 둔 가구에 하수도요금 감면,95.652174
101,경남,1. 함께 일하고 함께 돌보는 사회(자체),신혼부부 주거자금 대출이자 지원사업,50.0,지원대상 : 혼인신고~신청일 현재 5년이내신혼 부부 지원내용:주거자금대출이자지원,신혼부부 주거자금 대출이자 지원,50.0,1년 이상 관내 거주 신혼 부부 대출 잔액의 1.5% 이자 지원,94.444444
65,경기,1. 함께 일하고 함께 돌보는 사회(자체),다자녀 가정 상수도 요금 감면,0.0,수도요금 감면을 통한 다자녀 가구 지원,다자녀 가정 하수도 요금 감면,0.0,⌜주민등록법⌟상 18세 이하의 자녀가 3명이상 동일세대로 구성되어 있는 가구 등에 대한 하수 도 요금감면,93.750000
209,충북,1. 함께 일하고 함께 돌보는 사회(자체),산모·신생아 건강관리 지원,40.0,"지원대상 : 임산부 지원내용:본인부담금의90%지원(상한금50만원) 지원기준:산모가출산(예정)일기준으로6개월전부 터보은군에주민등록을두었으며,신생아출생신고를 군에한경우",산모 신생아 건강관리 지원,40.0,"지원대상 : 신생아 출생일 기준 부 또는 모가 1년 이전부터 계속하여 거주하는 관내 임산부 내용 산모신생아 건강관리 지원사업 본인부담금의 90%지원 기준중위소득 150% 초과 가정이면서, 지원 자격 충족시 총 서비스 금액90% 지원",92.857143
45,경기,1. 함께 일하고 함께 돌보는 사회(자체),다자녀가구 공영주차장 주차요금감면,0.0,지원대상 : 두자녀이상 자녀를 둔 가정 지원내용:관내공영주차장주차요금50%감면,다자녀가정 공영주차장 주차요금 감면,0.0,저출산·고령사회 기본법 제10조에 따라 두자녀 이상으로 표기한 다자녀가정 우대카드(I-plus카드)를소지한차량(최초2시간무 료후주차요금50퍼센트경감),91.891892
178,전남,1. 함께 일하고 함께 돌보는 사회(자체),난임 진단 검진비 지원,15.0,지원대상:관내 난임진단을 받은 법적 혼인부 부 지원내용 : 난임 진단검진비 최대30만원(부부 합산)1회 지원,난임 진단 검사비 지원,15.0,NaN,91.666667
138,부산,1. 함께 일하고 함께 돌보는 사회(자체),임산부 출산교실 운영,3.0,임산부 출산교실 운영,임산부 출산준비교실 운영,3.0,"지원대상: 관내 임산부 지원내용:임신,출산,육아에대한정보제공",91.666667
175,인천,1. 함께 일하고 함께 돌보는 사회(자체),다자녀 가정 병원 진료비 감면(계양구),0.0,지원대상 : 둘째 이상 다자녀 가정 지원내용:협약기관이용시진료비감면,다자녀 가정 병원 진료비 감면(부평구),0.0,지원대상 : 세자녀 이상 다자녀 가정 지원내용:협약기관이용시진료비감면,90.476190


In [29]:
# 당해예산 0.0이 우연일치인지 확인
print("당해예산 = 0 으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] == 0).sum())
print("0이 아닌 값으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] != 0).sum())

near_dup_confidenet = near_dup_same_budget[near_dup_same_budget["당해예산1"] != 0]
display(near_dup_confidenet)

당해예산 = 0 으로 일치한 건수:  7
0이 아닌 값으로 일치한 건수:  4


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
101,경남,1. 함께 일하고 함께 돌보는 사회(자체),신혼부부 주거자금 대출이자 지원사업,50.0,지원대상 : 혼인신고~신청일 현재 5년이내신혼 부부 지원내용:주거자금대출이자지원,신혼부부 주거자금 대출이자 지원,50.0,1년 이상 관내 거주 신혼 부부 대출 잔액의 1.5% 이자 지원,94.444444
209,충북,1. 함께 일하고 함께 돌보는 사회(자체),산모·신생아 건강관리 지원,40.0,"지원대상 : 임산부 지원내용:본인부담금의90%지원(상한금50만원) 지원기준:산모가출산(예정)일기준으로6개월전부 터보은군에주민등록을두었으며,신생아출생신고를 군에한경우",산모 신생아 건강관리 지원,40.0,"지원대상 : 신생아 출생일 기준 부 또는 모가 1년 이전부터 계속하여 거주하는 관내 임산부 내용 산모신생아 건강관리 지원사업 본인부담금의 90%지원 기준중위소득 150% 초과 가정이면서, 지원 자격 충족시 총 서비스 금액90% 지원",92.857143
178,전남,1. 함께 일하고 함께 돌보는 사회(자체),난임 진단 검진비 지원,15.0,지원대상:관내 난임진단을 받은 법적 혼인부 부 지원내용 : 난임 진단검진비 최대30만원(부부 합산)1회 지원,난임 진단 검사비 지원,15.0,NaN,91.666667
138,부산,1. 함께 일하고 함께 돌보는 사회(자체),임산부 출산교실 운영,3.0,임산부 출산교실 운영,임산부 출산준비교실 운영,3.0,"지원대상: 관내 임산부 지원내용:임신,출산,육아에대한정보제공",91.666667


# 주요내용 구조 패턴 검사 
- 원자화를 위해 '지원대상:..', '지원내용: ..' 이 모든 행에 포함되어있는지 확인한다. 

In [30]:
def dedup_label(text: str) -> str:
    """지원대상 : 지원대상 : ...  처럼 같은 라벨이 연속으로 중복된 걸 하나로 정리"""
    if pd.isna(text):
        return text
    text = re.sub(
        r"\(\s*(지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\s*\)",
        r"\1 : ",
        text,
    )
    text = re.sub(r"(지원대상\s*[:：]\s*)+", "지원대상 : ", text)
    text = re.sub(r"(지원내용\s*[:：]\s*)+", "지원내용 : ", text)
    text = re.sub(r"(사업대상\s*[:：]\s*)+", "사업대상 : ", text)
    text = re.sub(r"(사업내용\s*[:：]\s*)+", "사업내용 : ", text)
    return text


df_leaf_final["주요내용"] = df_leaf_final["주요내용"].apply(dedup_label)

In [31]:
support_pattern = re.compile(r"^지원대상\s*[:：]\s*(.*?)\s*지원내용\s*[:：]\s*(.*)$", re.DOTALL)


def check_pattern(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern.match(text) else "불일치"


df_leaf_final["주요내용_패턴"] = df_leaf_final["주요내용"].apply(check_pattern)

print(df_leaf_final["주요내용_패턴"].value_counts())
print(df_leaf_final["주요내용_패턴"].value_counts(normalize=True).mul(100).round(1))

주요내용_패턴
불일치    5239
일치     1782
결측      133
Name: count, dtype: int64
주요내용_패턴
불일치    73.2
일치     24.9
결측      1.9
Name: proportion, dtype: float64


In [32]:
# 범위 넓혀보기
support_pattern_broad = re.compile(
    r"^(지원대상|사업대상|대상)\s*[:：]?\s*(.*?)\s*(지원내용|사업내용|내용)\s*[:：]?\s*(.*)$",
    re.DOTALL,
)


def check_pattern_broad(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern_broad.match(text) else "불일치"


df_leaf_final["주요내용_패턴_확장"] = df_leaf_final["주요내용"].apply(check_pattern_broad)

print(df_leaf_final["주요내용_패턴_확장"].value_counts())
print(df_leaf_final["주요내용_패턴_확장"].value_counts(normalize=True).mul(100).round(1))

# 여전히 안걸리는 샘플 확인
display(
    df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"][
        ["지역", "세부사업명", "주요내용"]
    ].head(20)
)

주요내용_패턴_확장
불일치    4850
일치     2171
결측      133
Name: count, dtype: int64
주요내용_패턴_확장
불일치    67.8
일치     30.3
결측      1.9
Name: proportion, dtype: float64


,지역,세부사업명,주요내용
34,서울,방과후학교 운영 지원,"방과후학교 사업지원: 공립초, 공∙사립중, 공∙ 사립고, 특성화고(특목고 제외) 지원 방과후돌봄지원센터구축∙운영 방과후학교지원센터운영"
35,서울,초등돌봄교실운영,초등돌봄교실 사업지원: 공립초 전체 초등돌봄교실운영시간확대 돌봄이용모든학생에간식지원 지역사회연계한돌봄서비스강화 돌봄교실운영확대지원(46실증설)
39,서울,노인맞춤 돌봄서비스,"지원대상 : 만 65세 이상 국민기초생활수급자, 차상위계층, 기초연금 수급자 중 돌봄이 필요한 어르신 대상자의욕구를반영하여각분야별서비스동시제공"
40,서울,예방접종 관리,공공실버케어센터 확충
62,서울,육아휴직 활성화(교원),육아휴직 활성화(교원)
63,서울,육아휴직 활성화(지방공무원),"6개월 이상(출산휴가 연계 시 3개월 이상) 육 아휴직 시 후임 충원, 둘째 이후 자녀의 육아 휴직 시 전 기간 100% 경력(승진소요 최저연 수, 경력평정) 인정"
64,서울,다자녀 여성 지방공무원 탄력근무제 운영,다자녀 여성 지방공무원 탄력근무(출퇴근 전후 30분~1시간 단위) 실시
65,서울,교육공무원의 육아기 근로시간 단축제도 운영,교육공무원의 육아기 근로시간 단축제도 운영
66,서울,출산·양육 교직원 인사 우대정책 강화(지방공무원),다자녀 여성 지방공무원 탄력근무(출퇴근 전후 30분~1시간 단위) 실시
67,서울,출산·양육 교직원 인사 우대정책 강화(교원),출산·양육 교직원 인사 우대정책 강화(교원)


In [33]:
def extract_via_regex(text: str) -> dict:
    """패턴에 걸리면 그대로 분리, 안 걸리면 None"""
    if pd.isna(text):
        return {"지원대상": None, "지원내용": None}
    m = support_pattern_broad.match(text)
    if m:
        return {"지원대상": m.group(2).strip(), "지원내용": m.group(4).strip()}
    return {"지원대상": None, "지원내용": None}


regex_extracted = pd.json_normalize(df_leaf_final["주요내용"].apply(extract_via_regex))
df_leaf_final["지원대상"] = regex_extracted["지원대상"]
df_leaf_final["지원내용_상세"] = regex_extracted["지원내용"]

In [34]:
df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"].head()

,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,당해예산,전년도예산,증감액,증감율,원본행,주요내용_패턴,주요내용_패턴_확장,지원대상,지원내용_상세
34,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,방과후학교 운영 지원,"방과후학교 사업지원: 공립초, 공∙사립중, 공∙ 사립고, 특성화고(특목고 제외) 지원 방과후돌봄지원센터구축∙운영 방과후학교지원센터운영",13560.0,14951.0,-1391.0,-9.3,45,불일치,불일치,NaN,NaN
35,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),NaN,초등돌봄교실운영,초등돌봄교실 사업지원: 공립초 전체 초등돌봄교실운영시간확대 돌봄이용모든학생에간식지원 지역사회연계한돌봄서비스강화 돌봄교실운영확대지원(46실증설),40237.0,26045.0,14192.0,54.5,46,불일치,불일치,NaN,NaN
39,2024,서울,Ⅰ. 공통사업,2. 건강하고 능동적인 고령사회(공통),공통,노인맞춤 돌봄서비스,"지원대상 : 만 65세 이상 국민기초생활수급자, 차상위계층, 기초연금 수급자 중 돌봄이 필요한 어르신 대상자의욕구를반영하여각분야별서비스동시제공",76.0,68.0,8.0,11.8,50,불일치,불일치,NaN,NaN
40,2024,서울,Ⅰ. 공통사업,2. 건강하고 능동적인 고령사회(공통),공통,예방접종 관리,공공실버케어센터 확충,77619.0,81358.0,-3739.0,-4.6,51,불일치,불일치,NaN,NaN
62,2024,서울,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),자체,육아휴직 활성화(교원),육아휴직 활성화(교원),0.0,85.0,-85.0,-100.0,77,불일치,불일치,NaN,NaN


# 주요내용 LLM 정제 (업스테이지 Solar Pro 3, 구조화 출력)

- LLM은 `주요내용_정제`(오탈자·이상한 공백만 보존형으로 교정) 생성만 담당한다. 요약/재구성 없음, 숫자·고유명사 변경 없음.
- `지원대상`/`지원내용_상세`는 이미 검증된 정규식(`extract_via_regex`) 결과를 그대로 쓴다 — 정규식으로 걸러지는 데 LLM을 또 쓰는 건 실익이 없고 과잉분리 위험만 늘린다.
- `response_format`(json_schema)으로 API 단에서 출력 구조를 강제, 실패 시 1회 재시도 후에도 실패하면 원문을 그대로 사용(정제문장 결측 방지).
- 정제 전후 숫자(금액·비율·연령 등) 보존 여부를 자동 검증해, 달라진 행은 검토 대상으로 표시한다.

In [35]:
import os
import json
import yaml
from openai import OpenAI
from tqdm import tqdm

tqdm.pandas()

with open("../configs/llm_extraction.yaml") as f:
    llm_cfg = yaml.safe_load(f)

client = OpenAI(
    api_key=os.environ["UPSTAGE_API_KEY"],
    base_url=llm_cfg["upstage"]["base_url"],
)

RESPONSE_FORMAT = {"type": "json_schema", "json_schema": llm_cfg["response_schema"]}


def call_llm_once(name: str, content: str) -> str | None:
    """API 1회 호출. 파싱 실패하면 None 반환"""
    prompt = llm_cfg["prompt"]["template"].format(name=name, content=content)
    response = client.chat.completions.create(
        model=llm_cfg["upstage"]["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=llm_cfg["upstage"]["temperature"],
        response_format=RESPONSE_FORMAT,
    )
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)["정제문장"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return None


def clean_sentence(name: str, content: str) -> str:
    """주요내용을 LLM으로 정제. 결측이면 결측 유지, 실패 시(1회 재시도 포함) 원문 그대로 사용"""
    if pd.isna(content):
        return None
    for attempt in range(2):  # 최초 시도 + 1회 재시도
        try:
            result = call_llm_once(name, content)
        except Exception as e:
            print(f"API 호출 실패({attempt + 1}회차): {name} -> {e}")
            result = None
        if result is not None:
            return result
    print(f"정제 실패, 원문 유지: {name}")
    return content


def extract_numbers(text) -> list:
    """숫자(금액·비율·연령 등) 시퀀스만 뽑아 리스트로 반환 - 원문/정제문장 보존 검증용"""
    if pd.isna(text):
        return []
    return re.findall(r"\d+", text)


def numbers_preserved(original, cleaned) -> bool:
    """정제 과정에서 숫자가 그대로 보존됐는지 확인 (다르면 검토 대상)"""
    return extract_numbers(original) == extract_numbers(cleaned)

# 먼저 소량 샘플로 속도·품질 확인 (전체 실행 전 확인용)

```python
sample = df_leaf_final.sample(
    llm_cfg["sample"]["size"], random_state=llm_cfg["sample"]["random_state"]
).copy()

sample["주요내용_정제"] = sample.progress_apply(
    lambda row: clean_sentence(row["세부사업명"], row["주요내용"]), axis=1
)
sample["숫자보존"] = sample.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~sample["숫자보존"]).sum())
display(
    sample[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세", "숫자보존"]]
)
```

샘플 결과가 괜찮으면 아래 셀로 전체 실행 (7천여 건, 시간 오래 걸릴 수 있음)

In [36]:
CHECKPOINT_PATH = INTERIM_DIR / "llm_정제_체크포인트.csv"
CHUNK_SIZE = 200

if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    done_index = set(checkpoint_df.index)
    print(f"체크포인트 발견: {len(done_index)}건 이미 처리됨, 이어서 진행")
else:
    checkpoint_df = pd.DataFrame(columns=["주요내용_정제"])
    done_index = set()

targets = df_leaf_final.index.difference(done_index)
results = {}

for i, idx in enumerate(tqdm(targets), start=1):
    row = df_leaf_final.loc[idx]
    results[idx] = clean_sentence(row["세부사업명"], row["주요내용"])

    if i % CHUNK_SIZE == 0:
        partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
        checkpoint_df = pd.concat([checkpoint_df, partial])
        checkpoint_df.to_csv(CHECKPOINT_PATH)
        results = {}

# 남은 것 마저 저장
if results:
    partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
    checkpoint_df = pd.concat([checkpoint_df, partial])
    checkpoint_df.to_csv(CHECKPOINT_PATH)

df_leaf_final["주요내용_정제"] = checkpoint_df["주요내용_정제"]
df_leaf_final["숫자보존"] = df_leaf_final.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~df_leaf_final["숫자보존"]).sum())
display(
    df_leaf_final[~df_leaf_final["숫자보존"]][
        ["지역", "세부사업명", "주요내용", "주요내용_정제"]
    ].head(20)
)

df_leaf_final[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세"]].head(20)

체크포인트 발견: 7154건 이미 처리됨, 이어서 진행


0it [00:00, ?it/s]

숫자 불일치(검토 대상) 건수: 0


,지역,세부사업명,주요내용,주요내용_정제


,세부사업명,주요내용,주요내용_정제,지원대상,지원내용_상세
2,저출생 극복 지역네트워크 구축사업 지원,지원대상 : 서울시민(3~7세 자녀가 있는 가정) 지원내용 : 서울100인의아빠단,지원대상 : 서울시민(3~7세 자녀가 있는 가정) 지원내용 : 서울100인의아빠단,서울시민(3~7세 자녀가 있는 가정),서울100인의아빠단
3,입양아동 가족지원,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용 : 입양아동양육보조금,양육수당,입양비용등","지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용 : 입양아동양육보조금, 양육수당, 입양비용등",입양특례법에 의한 18세미만 입양 아동 및 그 가정,"입양아동양육보조금,양육수당,입양비용등"
4,국공립어린이집 등 보육서비스 수준제고,지원대상 : 국공립 어린이집 등 지원내용 : 보육교직원인건비지원,지원대상 : 국공립 어린이집 등 지원내용 : 보육교직원인건비지원,국공립 어린이집 등,보육교직원인건비지원
5,어린이집 이용 영유아 무상보육 확대,지원대상 : 만0~2세 영아 지원내용 : 어린이집이용영아대상바우처지원,지원대상 : 만0~2세 영아 지원내용 : 어린이집이용영아대상바우처지원,만0~2세 영아,어린이집이용영아대상바우처지원
6,초등돌봄 공적 인프라 확충,지원대상 : 6~12세 아동 지원내용 : 돌봄서비스제공,지원대상 : 6~12세 아동 지원내용 : 돌봄서비스제공,6~12세 아동,돌봄서비스제공
7,육아종합지원센터 운영,"지원대상 : 어린이집 및 영유아 양육 가정 지원내용 어린이집 지원사업 : 어린이집 맞춤 컨설팅, 장애아지원 교육 보육교직원 교육 및 상담, 서울형 모아어린이집 운영지원 등 가정양육 지원사업 : 장난감도서관 운영, 시 간제보육 관리등","지원대상 : 어린이집 및 영유아 양육 가정 지원내용 어린이집 지원사업 : 어린이집 맞춤 컨설팅, 장애아지원 교육 보육교직원 교육 및 상담, 서울형 모아어린이집 운영지원 등 가정양육 지원사업 : 장난감도서관 운영, 시간제보육 관리등",어린이집 및 영유아 양육 가정,"어린이집 지원사업 : 어린이집 맞춤 컨설팅, 장애아지원 교육 보육교직원 교육 및 상담, 서울형 모아어린이집 운영지원 등 가정양육 지원사업 : 장난감도서관 운영, 시 간제보육 관리등"
8,가정양육수당 지원,지원대상 : 시설 미이용 24개월~86개월 미만 아동 지원내용 : 월10만원(장애아동35개월미만20만원) 지원,지원대상 : 시설 미이용 24개월~86개월 미만 아동 지원내용 : 월10만원(장애아동35개월미만20만원) 지원,시설 미이용 24개월~86개월 미만 아동,월10만원(장애아동35개월미만20만원) 지원
9,부모급여,"지원대상 : 0~1세 아동 지원내용 : 0세100만원(월),1세50만원(월)지원","지원대상 : 0~1세 아동 지원내용 : 0세 100만원(월), 1세 50만원(월) 지원",0~1세 아동,"0세100만원(월),1세50만원(월)지원"
10,공동육아나눔터,"지원대상 : 자녀를 둔 가정(자녀 및 보호자) 지원내용 : 공동육아공간제공,가족품앗이활동지원","지원대상 : 자녀를 둔 가정(자녀 및 보호자) 지원내용 : 공동육아공간제공,가족품앗이활동지원",자녀를 둔 가정(자녀 및 보호자),"공동육아공간제공,가족품앗이활동지원"
11,아이돌봄서비스 확충 및 내실화,지원대상 : 맞벌이 등 양육공백 가정의 만 12 세 이하 아동 지원내용 : 아이돌보미를연계해찾아가는아이돌봄 서비스제공,지원대상 : 맞벌이 등 양육공백 가정의 만 12 세 이하 아동 지원내용 : 아이돌보미를 연계해 찾아가는 아이돌봄 서비스 제공,맞벌이 등 양육공백 가정의 만 12 세 이하 아동,아이돌보미를연계해찾아가는아이돌봄 서비스제공


In [37]:
PUA_LOW, PUA_HIGH = chr(0xE000), chr(0xF8FF)
pua_re = re.compile(f"[{PUA_LOW}-{PUA_HIGH}]")
paren_label_re = re.compile(
    r"\((지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\)"
)


def needs_llm_rerun(text):
    if pd.isna(text):
        return False
    return bool(pua_re.search(text)) or bool(paren_label_re.search(text))


affected_mask = df_leaf_final["주요내용_정제"].apply(needs_llm_rerun)
affected_idx = df_leaf_final.index[affected_mask]
print("재실행 대상 행수:", len(affected_idx))

# 체크포인트에서 해당 행 제거 -> 다음에 위 LLM 병합 셀을 다시 실행하면 이 행들만 API 재호출됨
checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
checkpoint_df = checkpoint_df.drop(index=affected_idx, errors="ignore")
checkpoint_df.to_csv(CHECKPOINT_PATH)
print("체크포인트에서 제거:", len(affected_idx), "건 -> 남은 행수:", len(checkpoint_df))

재실행 대상 행수: 0
체크포인트에서 제거: 0 건 -> 남은 행수: 7154


In [38]:
output_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "주요내용_정제",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]

missing_cols = [c for c in output_cols if c not in df_leaf_final.columns]

if missing_cols:
    raise KeyError(f"df_leaf_final에 없는 칼럼: {missing_cols}")

for sido_name, group in df_leaf_final.groupby("지역"):
    out_path = INTERIM_DIR / sido_name / f"2024_{sido_name}_세부사업_정제.csv"
    group[output_cols].to_csv(out_path, index=False)
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")


print("\n숫자보존 불일치 총 건수:", (~df_leaf_final["숫자보존"]).sum())

강원: 496행 저장 -> ../data/interim/강원/2024_강원_세부사업_정제.csv
경기: 1315행 저장 -> ../data/interim/경기/2024_경기_세부사업_정제.csv
경남: 714행 저장 -> ../data/interim/경남/2024_경남_세부사업_정제.csv
경북: 694행 저장 -> ../data/interim/경북/2024_경북_세부사업_정제.csv
광주: 200행 저장 -> ../data/interim/광주/2024_광주_세부사업_정제.csv
대구: 253행 저장 -> ../data/interim/대구/2024_대구_세부사업_정제.csv
대전: 162행 저장 -> ../data/interim/대전/2024_대전_세부사업_정제.csv
부산: 683행 저장 -> ../data/interim/부산/2024_부산_세부사업_정제.csv
서울: 237행 저장 -> ../data/interim/서울/2024_서울_세부사업_정제.csv
세종: 120행 저장 -> ../data/interim/세종/2024_세종_세부사업_정제.csv
울산: 302행 저장 -> ../data/interim/울산/2024_울산_세부사업_정제.csv
인천: 328행 저장 -> ../data/interim/인천/2024_인천_세부사업_정제.csv
전남: 468행 저장 -> ../data/interim/전남/2024_전남_세부사업_정제.csv
전북: 419행 저장 -> ../data/interim/전북/2024_전북_세부사업_정제.csv
제주: 156행 저장 -> ../data/interim/제주/2024_제주_세부사업_정제.csv
충남: 106행 저장 -> ../data/interim/충남/2024_충남_세부사업_정제.csv
충북: 501행 저장 -> ../data/interim/충북/2024_충북_세부사업_정제.csv

숫자보존 불일치 총 건수: 0


In [39]:
df_leaf_final.columns.tolist()

['연도',
 '지역',
 '대분류',
 '중분류',
 '사업분류재정구분',
 '세부사업명',
 '주요내용',
 '당해예산',
 '전년도예산',
 '증감액',
 '증감율',
 '원본행',
 '주요내용_패턴',
 '주요내용_패턴_확장',
 '지원대상',
 '지원내용_상세',
 '주요내용_정제',
 '숫자보존']

In [40]:
id_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
    "주요내용_정제",
]

df_long = df_leaf_final.melt(
    id_vars=id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)

# 전년도예산 행은 실제 그 돈이 집행된 연도로 맞춰서 -1
previous_mask = df_long["예산구분"] == "전년도예산"
df_long.loc[previous_mask, "연도"] -= 1
# 증감액/증감율은 "당해 대비 전년" 개념이라 전년도 행에는 그대로 복제되면 안 됨 (CodeRabbit 지적)
df_long.loc[previous_mask, ["증감액", "증감율"]] = None

df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

print("long 변환 결과:", len(df_long), "행 (wide", len(df_leaf_final), "행 x 2)")
df_long.head(10)

long 변환 결과: 14308 행 (wide 7154 행 x 2)


,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,증감액,증감율,원본행,지원대상,지원내용_상세,주요내용_정제,예산구분,예산액
0,2024,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,아이돌봄 지원,맞벌이등 양육공백가정에 아이돌봄서비스 제공,8353.0,32.1,4157,NaN,NaN,맞벌이등 양육공백가정에 아이돌봄서비스 제공,당해예산,34396.0
1,2023,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,아이돌봄 지원,맞벌이등 양육공백가정에 아이돌봄서비스 제공,NaN,NaN,4157,NaN,NaN,맞벌이등 양육공백가정에 아이돌봄서비스 제공,전년도예산,26043.0
2,2024,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,아동비만예방 '건강한 돌봄놀이터'운영,지원대상 : 돌봄놀이터 신청 아동 120명 지원내용 : 강사수당및운영비,4.0,133.3,4158,돌봄놀이터 신청 아동 120명,강사수당및운영비,지원대상 : 돌봄놀이터 신청 아동 120명 지원내용 : 강사수당및운영비,당해예산,7.0
3,2023,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,아동비만예방 '건강한 돌봄놀이터'운영,지원대상 : 돌봄놀이터 신청 아동 120명 지원내용 : 강사수당및운영비,NaN,NaN,4158,돌봄놀이터 신청 아동 120명,강사수당및운영비,지원대상 : 돌봄놀이터 신청 아동 120명 지원내용 : 강사수당및운영비,전년도예산,3.0
4,2024,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,지역급식관리지원 센터 운영,"지원대상 : 원주시 어린이 사회복지 급식관리지 원센터에 등록되어있는 어린이급식소 및 사회복 지 급식소 지원내용 : 어린이급식소,사회복지급식소위생,영양 관리순회방문지도및급식지원등",834.0,NaN,4159,원주시 어린이 사회복지 급식관리지 원센터에 등록되어있는 어린이급식소 및 사회복 지 급식소,"어린이급식소,사회복지급식소위생,영양 관리순회방문지도및급식지원등","지원대상 : 원주시 어린이 사회복지 급식관리지원 센터에 등록되어있는 어린이급식소 및 사회복지 급식소 지원내용 : 어린이급식소,사회복지 급식소 위생,영양 관리 순회방문지도 및 급식지원 등",당해예산,834.0
5,2023,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,지역급식관리지원 센터 운영,"지원대상 : 원주시 어린이 사회복지 급식관리지 원센터에 등록되어있는 어린이급식소 및 사회복 지 급식소 지원내용 : 어린이급식소,사회복지급식소위생,영양 관리순회방문지도및급식지원등",NaN,NaN,4159,원주시 어린이 사회복지 급식관리지 원센터에 등록되어있는 어린이급식소 및 사회복 지 급식소,"어린이급식소,사회복지급식소위생,영양 관리순회방문지도및급식지원등","지원대상 : 원주시 어린이 사회복지 급식관리지원 센터에 등록되어있는 어린이급식소 및 사회복지 급식소 지원내용 : 어린이급식소,사회복지 급식소 위생,영양 관리 순회방문지도 및 급식지원 등",전년도예산,0.0
6,2024,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,임산부 등록 건강관리,지원대상 : 임산부 및 임신 예비 여성 지원내용 : 임산부영양제제공등,39.0,57.4,4160,임산부 및 임신 예비 여성,임산부영양제제공등,지원대상 : 임산부 및 임신 예비 여성 지원내용 : 임산부영양제제공등,당해예산,107.0
7,2023,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,임산부 등록 건강관리,지원대상 : 임산부 및 임신 예비 여성 지원내용 : 임산부영양제제공등,NaN,NaN,4160,임산부 및 임신 예비 여성,임산부영양제제공등,지원대상 : 임산부 및 임신 예비 여성 지원내용 : 임산부영양제제공등,전년도예산,68.0
8,2024,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,출생아 건강관리〮 보험 료 지원,셋째아 이후 출생아 1인당 200만원 이하 보험 비 및 건강관리비 지원,1.0,0.5,4161,NaN,NaN,셋째아 이후 출생아 1인당 200만원 이하 보험비 및 건강관리비 지원,당해예산,198.0
9,2023,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,출생아 건강관리〮 보험 료 지원,셋째아 이후 출생아 1인당 200만원 이하 보험 비 및 건강관리비 지원,NaN,NaN,4161,NaN,NaN,셋째아 이후 출생아 1인당 200만원 이하 보험비 및 건강관리비 지원,전년도예산,197.0


In [41]:
# 시도별로 저장
for sido_name, group in df_long.groupby("지역"):
    out_path = INTERIM_DIR / sido_name / f"2024_{sido_name}_세부사업_정제_long.csv"
    group.to_csv(out_path, index=False)
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")

강원: 992행 저장 -> ../data/interim/강원/2024_강원_세부사업_정제_long.csv
경기: 2630행 저장 -> ../data/interim/경기/2024_경기_세부사업_정제_long.csv
경남: 1428행 저장 -> ../data/interim/경남/2024_경남_세부사업_정제_long.csv
경북: 1388행 저장 -> ../data/interim/경북/2024_경북_세부사업_정제_long.csv
광주: 400행 저장 -> ../data/interim/광주/2024_광주_세부사업_정제_long.csv
대구: 506행 저장 -> ../data/interim/대구/2024_대구_세부사업_정제_long.csv
대전: 324행 저장 -> ../data/interim/대전/2024_대전_세부사업_정제_long.csv
부산: 1366행 저장 -> ../data/interim/부산/2024_부산_세부사업_정제_long.csv
서울: 474행 저장 -> ../data/interim/서울/2024_서울_세부사업_정제_long.csv
세종: 240행 저장 -> ../data/interim/세종/2024_세종_세부사업_정제_long.csv
울산: 604행 저장 -> ../data/interim/울산/2024_울산_세부사업_정제_long.csv
인천: 656행 저장 -> ../data/interim/인천/2024_인천_세부사업_정제_long.csv
전남: 936행 저장 -> ../data/interim/전남/2024_전남_세부사업_정제_long.csv
전북: 838행 저장 -> ../data/interim/전북/2024_전북_세부사업_정제_long.csv
제주: 312행 저장 -> ../data/interim/제주/2024_제주_세부사업_정제_long.csv
충남: 212행 저장 -> ../data/interim/충남/2024_충남_세부사업_정제_long.csv
충북: 1002행 저장 -> ../data/interim/충북/2024_충북_세부사업_정제_l